In [7]:
import pandas as pd
import numpy as np
import seaborn as sns

pol=pd.read_csv('../Personal Auto Policy Data and Rate Tables/Personal Auto Dataset and Premiums.csv')
pol.columns=pol.columns.str.lower().str.replace(' ','_')
tables=pd.read_excel('rate_tables.xlsx',sheet_name=None)

rated=pol.copy() #copy the policy table for rating
rated.loc[:,'policy']=rated.index.to_list() #policyid
rated=rated.dropna(subset=['vehicle_symbol','vendor_mileage']) #five instances of nan rating variables
rated.loc[:,'f_base_rate']=tables['base_rate'].base_rate[0] #set the base rate
#merge basic rating tables onto rating variables
for t in ['vendor_mileage','homeowner','passive_restraint',
          'tnc_coverage','vehicle_symbol','claims_free_years',
          'good_student','convictions','driver_training']:
    rated=rated.merge(tables[t],how='left',on=t)

#nearest merge on vehicle age to handle ages above the table maximum
rated=pd.merge_asof(
    rated.sort_values('vehicle_age'),
    tables['vehicle_age'],
    on='vehicle_age',
    direction='backward')

#driver class merges on the intersection of three variables
rated=rated.merge(
    tables['driver_class'],how='left',
    on=['policyholder_age','marital_status','policyholder_sex'])

#uw_points is a calculated field before merging to the uw_points rating table
rated.loc[:,'uw_points']=1+rated[
    ['claims_free_points','good_student_points','convictions_points','driver_training_points']
    ].sum(axis=1)
rated=rated.merge(tables['uw_points'],how='left',on='uw_points')

#multiplicative rating algorithm to arrive at pure premium
rated.loc[:,'pp']=rated[[f for f in rated.columns if f.startswith('f_')]].product(axis=1)
rated.loc[:,'expenses']=rated.pp*0.2+30

rated.to_csv('rated_policies.zip', index=False, compression={'method': 'zip', 'archive_name': 'rated_policies.csv'})


In [6]:
import pandas as pd
import numpy as np
import datetime as dt

loss = rated[rated.coll_loss > 0].copy()
loss['eff_dt'] = pd.to_datetime(loss.policyeffectivedate)
loss['py'] = loss.eff_dt.dt.year
loss['exp_dt'] = loss.eff_dt + pd.offsets.DateOffset(years=1) - pd.Timedelta(days=1)
loss['ly_start'] = pd.to_datetime(loss['loss_year'].astype(str) + '-01-01')
loss['ly_end'] = pd.to_datetime(loss['loss_year'].astype(str) + '-12-31')
loss['valid_start'] = np.maximum(loss['eff_dt'], loss['ly_start'])
loss['valid_end'] = np.minimum(loss['exp_dt'], loss['ly_end'])

diff = (loss['valid_end'] - loss['valid_start']).dt.days
loss['loss_dt'] = loss['valid_start'] + pd.to_timedelta(np.random.rand(len(loss)) * diff, unit='D')
loss['start_month'] = loss['loss_dt'].dt.to_period('M').dt.to_timestamp()

master_months = pd.date_range(start='2020-01-01', end='2025-12-01', freq='MS')
calendar = pd.DataFrame({'eval_date': master_months, 'key': 1})
loss['key'] = 1

expanded = pd.merge(loss, calendar, on='key').drop('key', axis=1)
expanded = expanded[expanded['eval_date'] >= expanded['start_month']].copy()
expanded['maturity'] = (
    (expanded['eval_date'].dt.year - expanded['start_month'].dt.year) * 12 +
    (expanded['eval_date'].dt.month - expanded['start_month'].dt.month)
)

unique_claims = expanded[['policy', 'coll_loss']].drop_duplicates()
n_claims = len(unique_claims)
ids = unique_claims['policy'].values
ultimates = unique_claims['coll_loss'].values

n_slots = 6 
timings = (np.random.beta(1, 3, size=(n_claims, n_slots)) * 35 + 1).astype(int)
alphas = 1 + (timings / 36) * 5
gamma_samples = np.random.gamma(alphas, 1)
weights = gamma_samples / gamma_samples.sum(axis=1, keepdims=True)
payment_amounts = weights * ultimates[:, np.newaxis]

pay_matrix = np.zeros((n_claims, 37))
for i in range(n_claims):
    for slot in range(n_slots):
        m = timings[i, slot]
        pay_matrix[i, m:] += payment_amounts[i, slot]

lookup = {cid: pay_matrix[i] for i, cid in enumerate(ids)}

expanded['paid'] = [lookup[cid][int(min(m, 36))] if m > 0 else 0.0 
                   for cid, m in zip(expanded['policy'], expanded['maturity'])]

expanded['reporting_pct'] = 1 - (0.8 ** (expanded['maturity'] + 1))
noise = np.random.uniform(0.8, 1.2, size=len(expanded))
expanded['incurred'] = expanded['paid'] + (expanded['coll_loss'] - expanded['paid']) * expanded['reporting_pct'] * noise
expanded['incurred'] = np.maximum(expanded['incurred'], expanded['paid'])

results = expanded[['policy', 'maturity', 'loss_dt', 'eval_date', 'paid', 'incurred']].copy()
results = results.sort_values(by=['policy', 'eval_date'])
results['aym'] = results['loss_dt'].dt.to_period('M').dt.to_timestamp()


results.to_csv('loss_development.zip', index=False, compression={'method': 'zip', 'archive_name': 'loss_development.csv'})



In [21]:
import chainladder as cl

triangle = cl.Triangle(
    results[results.eval_date.dt.year<=2023],
    origin='aym',
    development='eval_date',
    columns=['paid', 'incurred']
)

links = triangle.link_ratio
dev = cl.Development(n_periods=24, average='volume')
dev.fit(triangle)
weighted_avg_ldfs = dev.ldf_

C:\Users\seant\AppData\Local\Programs\Python\Python314\Lib\site-packages\chainladder\core\triangle.py:246: UserWarning: 
                The cumulative property of your triangle is not set. This may result in
                undesirable behavior. In a future release this will result in an error.
                
  warnings.warn(


In [22]:
weighted_avg_ldfs['paid']

,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,10-11,...,38-39,39-40,40-41,41-42,42-43,43-44,44-45,45-46,46-47,47-48
(All),,3.0336,2.0142,1.6813,1.5108,1.4091,1.3398,1.2902,1.2522,1.2225,...,1.0377,1.0363,1.0350,1.0339,1.0327,1.0317,1.0307,1.0298,1.0290,1.0281


In [23]:
weighted_avg_ldfs['incurred']

,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,10-11,...,38-39,39-40,40-41,41-42,42-43,43-44,44-45,45-46,46-47,47-48
(All),2.9514,1.9108,1.5756,1.4145,1.3193,1.2573,1.2136,1.1817,1.1573,1.1382,...,1.0288,1.0280,1.0272,1.0265,1.0258,1.0251,1.0245,1.0239,1.0234,1.0228
